# Know Your Knee — EDA, DICOM Mining & Baseline

**RSNA Knee Abnormality Detection** — A comprehensive EDA covering label distributions, radiology report mining, NLP label extraction, DICOM metadata analysis, and a 2.5D CNN baseline that produces a submittable CSV.

**What's inside:**
1. Data Overview & Structure
2. Label Deep Dive (12 binary targets, imbalance, co-occurrence)
   - Labeling Bias Warning
   - Radiology Report Analysis (sample reports, clinical term frequency)
   - 2b. NLP Label Extraction — expand from 58 to 4,000+ labeled studies
3. Series & Study Analysis (planes, sequences)
4. DICOM Metadata Mining (pixel spacing, slice thickness, manufacturers, field strength)
5. Image Visualization (windowing, normal vs abnormal, montages)
6. Slice-Level Analysis
7. Train vs Test Distribution
8. Full DICOM Header Reference
9. Key Takeaways & Modeling Implications
10. Baseline Model & Submission (2.5D ResNet18, trained on NLP-expanded labels)

In [ ]:
!pip install matplotlib-venn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
import pydicom
import os
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

COLORS = sns.color_palette("husl", 12)

# Data directory — mounted via competition data source
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Competition data not found! Please add 'rsna-knee-abnormality-detection' "
        "as a data source in notebook settings."
    )

print(f"Data directory: {DATA_DIR}")
print(f"Contents: {sorted([f.name for f in DATA_DIR.iterdir()])}")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("Setup complete!")

---
# 1. Data Overview

Let's start by loading all the CSV files and understanding the data structure.

In [ ]:
# Load all CSV files
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\ntrain.csv:        {train_df.shape[0]:,} rows × {train_df.shape[1]} cols")
print(f"test.csv:         {test_df.shape[0]:,} rows × {test_df.shape[1]} cols")
print(f"train_series.csv: {train_series_df.shape[0]:,} rows × {train_series_df.shape[1]} cols")
print(f"test_series.csv:  {test_series_df.shape[0]:,} rows × {test_series_df.shape[1]} cols")
print(f"sample_sub.csv:   {sample_sub.shape[0]:,} rows × {sample_sub.shape[1]} cols")

# Count DICOM files
train_dcm_dir = DATA_DIR / "train_series"
test_dcm_dir = DATA_DIR / "test_series"

train_studies = sorted(train_dcm_dir.iterdir()) if train_dcm_dir.exists() else []
test_studies = sorted(test_dcm_dir.iterdir()) if test_dcm_dir.exists() else []

print(f"\nTrain studies (directories): {len(train_studies):,}")
print(f"Test studies (directories):  {len(test_studies):,}")

In [ ]:
# Peek at train.csv
print("train.csv — first 5 rows:")
print("-" * 60)
display(train_df.head())
print(f"\nColumns: {list(train_df.columns)}")
print(f"Missing values:\n{train_df.isnull().sum()}")

In [ ]:
# Peek at train_series.csv
print("train_series.csv — first 5 rows:")
print("-" * 60)
display(train_series_df.head())
print(f"\nColumns: {list(train_series_df.columns)}")
print(f"\nMissing values:\n{train_series_df.isnull().sum()}")

In [ ]:
# Sample submission format
print("sample_submission.csv:")
print("-" * 60)
display(sample_sub.head(15))
print(f"\nUnique studies in submission: {sample_sub.shape[0] // len(TARGETS) if len(TARGETS) > 0 else '?'}")
print(f"Prediction columns: {[c for c in sample_sub.columns if c != 'StudyInstanceUID']}")

---
# 2. Label Deep Dive

The competition asks us to predict **12 knee abnormalities** from MRI scans. Let's understand the label distribution, class imbalance, and multi-label co-occurrence patterns.

### Clinical context for the 12 targets:
| Category | Abnormality | What it is |
|----------|------------|------------|
| **Ligament** | ACL tear | Anterior cruciate ligament — stabilizes rotation |
| **Ligament** | MCL tear | Medial collateral ligament — stabilizes inner knee |
| **Meniscus** | Medial Meniscus tear | Inner cartilage cushion |
| **Meniscus** | Lateral Meniscus tear | Outer cartilage cushion |
| **Arthritis** | Medial OA | Osteoarthritis on inner compartment |
| **Arthritis** | Lateral OA | Osteoarthritis on outer compartment |
| **Arthritis** | PF OA | Patellofemoral osteoarthritis (kneecap) |
| **Other** | Effusion | Fluid buildup in joint space |
| **Other** | Synovitis | Inflammation of synovial membrane |
| **Other** | Baker's cyst | Fluid-filled cyst behind knee |
| **Trauma** | Contusion | Bone bruise from impact |
| **Trauma** | Fracture | Bone break |

In [ ]:
# Identify target columns (everything except StudyInstanceUID and Report)
target_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
print(f"Target columns ({len(target_cols)}): {target_cols}")

# Only labeled studies (many studies have NaN labels — they have reports but no annotations)
labeled_df = train_df.dropna(subset=target_cols, how='all').copy()
print(f"\nLabeled studies: {len(labeled_df):,} / {len(train_df):,} ({len(labeled_df)/len(train_df)*100:.1f}%)")
print(f"Unlabeled studies (report-only): {len(train_df) - len(labeled_df):,}")

# Class distribution (labeled subset only)
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle(f"Class Distribution for Each Abnormality (n={len(labeled_df)} labeled studies)", 
             fontsize=16, fontweight='bold', y=1.02)

for idx, (col, ax) in enumerate(zip(target_cols, axes.ravel())):
    counts = labeled_df[col].value_counts().sort_index()
    bars = ax.bar(counts.index.astype(str), counts.values, color=[COLORS[idx], '#ff6b6b'], 
                  edgecolor='white', linewidth=1.5)
    
    total = counts.sum()
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
                f'{val}\n({val/total*100:.1f}%)', ha='center', va='bottom', fontsize=9)
    
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Normal (0)', 'Abnormal (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# Prevalence rate summary — sorted
prevalence = labeled_df[target_cols].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(prevalence)), prevalence.values, color=sns.color_palette("RdYlGn_r", len(prevalence)))
ax.set_yticks(range(len(prevalence)))
ax.set_yticklabels(prevalence.index, fontsize=11)
ax.set_xlabel("Prevalence (fraction of positive cases)", fontsize=12)
ax.set_title("Abnormality Prevalence — How Imbalanced Is Each Target?", fontsize=14, fontweight='bold')

for i, (val, name) in enumerate(zip(prevalence.values, prevalence.index)):
    ax.text(val + 0.005, i, f'{val:.1%}', va='center', fontsize=10, fontweight='bold')

ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Balanced (50%)')
ax.legend()
plt.tight_layout()
plt.show()

print("\nImbalance ratio (negative:positive):")
for col in prevalence.index:
    pos = labeled_df[col].sum()
    neg = len(labeled_df) - pos
    print(f"  {col:20s}: {neg:,} : {pos:,}  (ratio {neg/max(pos,1):.1f}:1)")

### Multi-Label Co-occurrence
Knee injuries rarely happen in isolation. An ACL tear often comes with bone contusions and effusion. Let's visualize which abnormalities tend to co-occur.

In [ ]:
# Co-occurrence matrix — P(B=1 | A=1) for each pair
n_targets = len(target_cols)
cooccurrence = np.zeros((n_targets, n_targets))

for i, col_a in enumerate(target_cols):
    for j, col_b in enumerate(target_cols):
        mask = labeled_df[col_a] == 1
        if mask.sum() > 0:
            cooccurrence[i, j] = labeled_df.loc[mask, col_b].mean()

fig, ax = plt.subplots(figsize=(12, 10))
mask_diag = np.eye(n_targets, dtype=bool)
sns.heatmap(cooccurrence, xticklabels=target_cols, yticklabels=target_cols,
            annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white', ax=ax,
            mask=mask_diag, cbar_kws={'label': 'P(column | row)'})

ax.set_title("Multi-Label Co-occurrence: P(column abnormality | row abnormality present)",
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Given this is positive...", fontsize=11)
ax.set_ylabel("How often is this also positive?", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Number of abnormalities per study
labeled_df['n_abnormalities'] = labeled_df[target_cols].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of number of abnormalities
counts = labeled_df['n_abnormalities'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=sns.color_palette("viridis", len(counts)),
            edgecolor='white', linewidth=1)
for x, y in zip(counts.index, counts.values):
    axes[0].text(x, y + 5, f'{y}\n({y/len(labeled_df)*100:.1f}%)', ha='center', fontsize=9)
axes[0].set_xlabel("Number of abnormalities", fontsize=12)
axes[0].set_ylabel("Number of studies", fontsize=12)
axes[0].set_title("How many abnormalities per knee?", fontsize=13, fontweight='bold')

# Normal vs any abnormality
has_any = (labeled_df['n_abnormalities'] > 0).sum()
normal = (labeled_df['n_abnormalities'] == 0).sum()
axes[1].pie([normal, has_any], labels=['Completely Normal', 'At Least 1 Abnormality'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90,
            textprops={'fontsize': 12}, pctdistance=0.6,
            explode=[0.05, 0.05])
axes[1].set_title("Normal vs. Abnormal Knees", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nMean abnormalities per study: {labeled_df['n_abnormalities'].mean():.2f}")
print(f"Median: {labeled_df['n_abnormalities'].median():.0f}")
print(f"Max: {labeled_df['n_abnormalities'].max()}")

In [ ]:
# Correlation between targets (Phi coefficient = Pearson on binary)
corr = labeled_df[target_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask_upper = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-0.3, vmax=0.7, linewidths=0.5, linecolor='white',
            mask=mask_upper, ax=ax, square=True,
            cbar_kws={'label': 'Pearson Correlation (Phi Coefficient)'})
ax.set_title("Label Correlation Matrix (Phi Coefficient)", fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print strongest correlations
print("\nTop 10 strongest positive correlations:")
corr_pairs = []
for i in range(len(target_cols)):
    for j in range(i+1, len(target_cols)):
        corr_pairs.append((target_cols[i], target_cols[j], corr.iloc[i, j]))
corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for a, b, r in corr_pairs[:10]:
    print(f"  {a:20s} ↔ {b:20s}: r = {r:+.3f}")

### ⚠️ Labeling Bias Warning

The 58 labeled studies average **4.14 abnormalities per knee** — meaning nearly every labeled study has multiple conditions. This is **not representative** of the full training population (which includes normal knees). The gold labels are heavily biased toward pathological cases, likely because annotators prioritized complex studies.

**Implication:** Prevalence rates computed from gold labels alone (shown above) overestimate how common each condition really is. We need the radiology reports to get a better picture.

### Radiology Reports — What Do They Look Like?

Before we try to extract labels from reports, let's peek at what we're working with.

In [ ]:
# ── Sample reports ────────────────────────────────────────────────────────
reports = train_df['Report'].dropna()
print(f"Studies with reports: {len(reports):,} / {len(train_df):,} ({len(reports)/len(train_df)*100:.1f}%)")
print(f"Studies without reports: {train_df['Report'].isna().sum():,}")

# Show one report from a labeled study and one from an unlabeled study
labeled_with_report = train_df[train_df[target_cols].notna().any(axis=1) & train_df['Report'].notna()]
unlabeled_with_report = train_df[train_df[target_cols].isna().all(axis=1) & train_df['Report'].notna()]

print(f"\n{'='*60}")
print("SAMPLE REPORT — Labeled study (has gold labels):")
print(f"{'='*60}")
sample_labeled = labeled_with_report.iloc[0]
print(f"StudyID: {sample_labeled['StudyInstanceUID']}")
print(f"Labels: {[c for c in target_cols if sample_labeled[c] == 1]}")
print(f"Report:\n{sample_labeled['Report'][:500]}")

print(f"\n{'='*60}")
print("SAMPLE REPORT — Unlabeled study (report only, no gold labels):")
print(f"{'='*60}")
sample_unlabeled = unlabeled_with_report.iloc[0]
print(f"StudyID: {sample_unlabeled['StudyInstanceUID']}")
print(f"Report:\n{sample_unlabeled['Report'][:500]}")

In [ ]:
# ── Report statistics + clinical term frequency ──────────────────────────
import re as re_mod

# Report length distribution
report_lengths = reports.str.len()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(report_lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel("Report length (characters)", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)
axes[0].set_title("Report Length Distribution", fontsize=13, fontweight='bold')
axes[0].axvline(report_lengths.median(), color='red', linestyle='--', 
                label=f'Median: {report_lengths.median():.0f} chars')
axes[0].legend()

# Clinical term frequency across all reports
clinical_terms = {
    'ACL': r'\bacl\b|anterior cruciate',
    'MCL': r'\bmcl\b|medial collateral',
    'meniscus': r'meniscus|meniscal',
    'tear': r'\btear\b|\btorn\b',
    'effusion': r'effusion',
    'fracture': r'fracture',
    'osteoarthritis': r'osteoarthr|arthrosis|\boa\b',
    'contusion': r'contusion|bone bruise',
    'synovitis': r'synovitis|synovial',
    'edema': r'\bedema\b',
    'intact': r'\bintact\b',
    'normal': r'\bnormal\b',
    'no ': r'\bno\b',
    "baker's cyst": r"baker|popliteal cyst",
    'cartilage': r'cartilage|chondr',
    'ligament': r'ligament',
}

reports_lower = reports.str.lower()
term_counts = {term: reports_lower.str.contains(pattern, regex=True, na=False).sum() 
               for term, pattern in clinical_terms.items()}
term_counts = dict(sorted(term_counts.items(), key=lambda x: x[1], reverse=True))

colors_bar = sns.color_palette("viridis", len(term_counts))
axes[1].barh(range(len(term_counts)), list(term_counts.values()), color=colors_bar)
axes[1].set_yticks(range(len(term_counts)))
axes[1].set_yticklabels(list(term_counts.keys()), fontsize=10)
axes[1].set_xlabel("Number of reports containing term", fontsize=12)
axes[1].set_title("Clinical Term Frequency Across All Reports", fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

for i, (term, count) in enumerate(term_counts.items()):
    axes[1].text(count + len(reports)*0.005, i, f'{count:,} ({count/len(reports)*100:.0f}%)', 
                 va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nReport length — min: {report_lengths.min()}, max: {report_lengths.max():,}, "
      f"mean: {report_lengths.mean():.0f}, median: {report_lengths.median():.0f}")

---
# 2b. Radiology Report Mining — 75x Label Expansion

**The labeling problem:** Only **58 out of 4,407** training studies (1.3%) have gold-standard binary labels. The remaining 4,349 studies have **radiology reports** but no annotations. Training on just 58 studies is why our baseline AUC is ~0.554 (random chance).

**The fix:** Extract labels from the free-text reports using regex pattern matching. Top Kaggle competitors (Pilkwang Kim, 524 votes) use this approach and report **~82% agreement** with gold labels. It's noisy, but 4,000+ weakly-labeled studies beats 58 perfect ones.

**Our approach:**
1. For each of the 12 targets, define **positive** patterns (e.g., `acl tear`, `acl rupture`) and **negative** patterns (e.g., `no acl tear`, `acl intact`)
2. Check negation first — if a negative pattern matches, label = 0
3. If a positive pattern matches (and no negation), label = 1
4. If no mention at all, label = **NaN** (not 0 — absence of mention ≠ absence of condition)
5. Validate against the 58 gold labels to measure extraction quality
6. Merge: gold labels override NLP where available, NLP fills the rest

In [ ]:
import re

# ── Pattern definitions for each target ──────────────────────────────────
# Structure: target_name -> (positive_patterns, negative_patterns)
# Negative patterns are checked FIRST — if matched, label = 0 even if
# a positive pattern also matches (handles "no ACL tear" correctly).

NLP_PATTERNS = {
    'ACL': {
        'pos': [
            r'acl\s*(tear|rupture|rupt|torn|injur|disrupt|deficien|incomplet|partial|full[- ]?thickness)',
            r'anterior\s+cruciate\s+(ligament\s+)?(tear|rupture|torn|injur|disrupt)',
            r'(torn|ruptured|disrupted|injured)\s+acl',
            r'acl\s+(is\s+)?(absent|not\s+intact|not\s+seen|discontinu)',
        ],
        'neg': [
            r'(no|without|negative\s+for|absent)\s+[^.]{0,30}acl\s*(tear|rupture|injur)',
            r'acl\s+(is\s+)?(intact|normal|unremarkable|preserved|continuous)',
            r'(intact|normal|stable)\s+acl',
        ],
    },
    'MCL': {
        'pos': [
            r'mcl\s*(tear|rupture|rupt|torn|injur|sprain|strain|disrupt|partial|full[- ]?thickness)',
            r'medial\s+collateral\s+(ligament\s+)?(tear|rupture|torn|injur|sprain)',
            r'(torn|ruptured|disrupted|injured)\s+mcl',
        ],
        'neg': [
            r'(no|without|negative\s+for)\s+[^.]{0,30}mcl\s*(tear|rupture|injur|sprain)',
            r'mcl\s+(is\s+)?(intact|normal|unremarkable|preserved)',
            r'(intact|normal|stable)\s+mcl',
        ],
    },
    'Medial Meniscus': {
        'pos': [
            r'medial\s+meniscus\s*.{0,20}(tear|torn|rupture|degenerat|macerat|fray|cleavage|flap|bucket|displaced|extru)',
            r'(tear|torn|rupture).{0,20}medial\s+meniscus',
            r'mm\s+(tear|torn)',  # common abbreviation
            r'medial\s+meniscal\s+(tear|degenerat|fray|cleavage|flap)',
        ],
        'neg': [
            r'(no|without|negative\s+for)\s+[^.]{0,30}medial\s+meniscus\s*(tear|rupture)',
            r'(no|without)\s+[^.]{0,30}medial\s+meniscal\s*(tear)',
            r'medial\s+meniscus\s+(is\s+)?(intact|normal|unremarkable)',
        ],
    },
    'Lateral Meniscus': {
        'pos': [
            r'lateral\s+meniscus\s*.{0,20}(tear|torn|rupture|degenerat|macerat|fray|cleavage|flap|bucket|displaced|extru)',
            r'(tear|torn|rupture).{0,20}lateral\s+meniscus',
            r'lateral\s+meniscal\s+(tear|degenerat|fray|cleavage|flap)',
        ],
        'neg': [
            r'(no|without|negative\s+for)\s+[^.]{0,30}lateral\s+meniscus\s*(tear|rupture)',
            r'(no|without)\s+[^.]{0,30}lateral\s+meniscal\s*(tear)',
            r'lateral\s+meniscus\s+(is\s+)?(intact|normal|unremarkable)',
        ],
    },
    'Medial OA': {
        'pos': [
            r'medial\s+(compartment\s+)?(osteoarthr|arthrit|arthrosis|OA|degenerat|narrowing|cartilage\s+loss)',
            r'(osteoarthr|arthrit|degenerat).{0,30}medial\s+(compartment|femoral|tibial)',
            r'medial\s+(joint\s+space|compartment)\s+(narrowing|loss)',
        ],
        'neg': [
            r'(no|without|negative\s+for)\s+[^.]{0,30}medial\s+(compartment\s+)?(osteoarthr|arthrit|OA|degenerat|narrowing)',
            r'medial\s+compartment\s+(is\s+)?(normal|preserved|unremarkable)',
        ],
    },
    'Lateral OA': {
        'pos': [
            r'lateral\s+(compartment\s+)?(osteoarthr|arthrit|arthrosis|OA|degenerat|narrowing|cartilage\s+loss)',
            r'(osteoarthr|arthrit|degenerat).{0,30}lateral\s+(compartment|femoral|tibial)',
            r'lateral\s+(joint\s+space|compartment)\s+(narrowing|loss)',
        ],
        'neg': [
            r'(no|without|negative\s+for)\s+[^.]{0,30}lateral\s+(compartment\s+)?(osteoarthr|arthrit|OA|degenerat|narrowing)',
            r'lateral\s+compartment\s+(is\s+)?(normal|preserved|unremarkable)',
        ],
    },
    'PF OA': {
        'pos': [
            r'patellofemoral\s*(osteoarthr|arthrit|arthrosis|OA|degenerat|narrowing|cartilage\s+loss|chondr)',
            r'(PF|patello[- ]?femoral)\s+(compartment\s+)?(OA|osteoarthr|degenerat|arthrit|narrowing)',
            r'patell\w*\s+(cartilage\s+)?(loss|thinning|degenerat|chondr|fissur)',
        ],
        'neg': [
            r'(no|without|negative\s+for)\s+[^.]{0,30}patellofemoral\s*(osteoarthr|arthrit|OA|degenerat)',
            r'patellofemoral\s+(compartment\s+)?(is\s+)?(normal|preserved|unremarkable)',
        ],
    },
    'Effusion': {
        'pos': [
            r'(joint\s+)?effusion',
            r'(suprapatellar|joint)\s+(pouch\s+)?(fluid|effusion)',
            r'fluid\s+(in|within)\s+(the\s+)?(joint|knee|suprapatellar)',
            r'(moderate|large|small|mild|trace|significant)\s+effusion',
        ],
        'neg': [
            r'no\s+(joint\s+)?effusion',
            r'(no|without|absent|negative\s+for)\s+[^.]{0,15}(effusion|joint\s+fluid)',
            r'no\s+(significant\s+)?(joint\s+)?fluid',
        ],
    },
    'Synovitis': {
        'pos': [
            r'synovitis',
            r'synovial\s+(thicken|inflam|hypertroph|proliferat|enhance)',
        ],
        'neg': [
            r'(no|without|absent|negative\s+for)\s+[^.]{0,15}synovitis',
            r'no\s+[^.]{0,15}synovial\s+(thicken|inflam)',
        ],
    },
    "Baker's": {
        'pos': [
            r"baker'?s?\s*(cyst|'s)",
            r'popliteal\s+(cyst|fossa\s+cyst)',
            r'(cyst|collection)\s+(in|within|of)\s+(the\s+)?popliteal',
        ],
        'neg': [
            r"(no|without|absent|negative\s+for)\s+[^.]{0,15}baker'?s?\s*cyst",
            r'(no|without)\s+[^.]{0,15}popliteal\s+cyst',
        ],
    },
    'Contusion': {
        'pos': [
            r'(bone\s+)?contusion',
            r'bone\s+(bruise|marrow\s+edema|marrow\s+signal)',
            r'(contusion|bruise)\s+(of|in|at|within)',
            r'marrow\s+edema\s+(pattern|signal)',
        ],
        'neg': [
            r'(no|without|absent|negative\s+for)\s+[^.]{0,15}(contusion|bone\s+bruise)',
            r'no\s+[^.]{0,20}(marrow\s+edema|bone\s+marrow\s+signal)',
        ],
    },
    'Fracture': {
        'pos': [
            r'fracture',
            r'(tibial\s+plateau|femoral\s+condyle|patell\w*|fibula)\s+fracture',
            r'(stress|insufficiency|avulsion|osteochondral)\s+fracture',
        ],
        'neg': [
            r'(no|without|absent|negative\s+for)\s+[^.]{0,15}fracture',
            r'no\s+(acute\s+|visible\s+|definite\s+)?fracture',
            r'fracture\s+(is\s+)?(excluded|ruled\s+out|not\s+seen|absent)',
        ],
    },
}


def extract_labels_from_report(text):
    """Extract 12 binary labels from a radiology report using regex.
    
    Returns dict with target names as keys.
    Values: 1 (positive), 0 (negative/negated), NaN (not mentioned).
    """
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) < 10:
        return {t: np.nan for t in NLP_PATTERNS}
    
    text_lower = text.lower()
    results = {}
    
    for target, patterns in NLP_PATTERNS.items():
        # Check negation first
        neg_match = any(re.search(p, text_lower) for p in patterns['neg'])
        pos_match = any(re.search(p, text_lower) for p in patterns['pos'])
        
        if neg_match and not pos_match:
            results[target] = 0
        elif pos_match and not neg_match:
            results[target] = 1
        elif pos_match and neg_match:
            # Both matched — negation is deliberate, trust it.
            # e.g. "No ACL tear" triggers both pos (acl tear) and neg (no...acl tear).
            results[target] = 0
        else:
            # No mention at all — NaN, not 0
            results[target] = np.nan
    
    return results


# Quick sanity check
test_report = "ACL tear with associated bone contusion. No meniscal tear. Moderate joint effusion."
test_labels = extract_labels_from_report(test_report)
print("Sanity check on sample report:")
print(f"  Report: {test_report!r}")
print(f"  Extracted labels:")
for k, v in test_labels.items():
    marker = "+" if v == 1 else ("-" if v == 0 else "?")
    print(f"    [{marker}] {k}: {v}")
print(f"\nPatterns defined for {len(NLP_PATTERNS)} targets")

In [ ]:
# ── Apply NLP extraction to ALL training studies ─────────────────────────
nlp_labels = train_df['Report'].apply(extract_labels_from_report)
nlp_df = pd.DataFrame(nlp_labels.tolist(), index=train_df.index)

# How many studies got at least one label?
has_any_nlp = nlp_df.notna().any(axis=1).sum()
has_no_report = train_df['Report'].isna().sum()

print(f"Total training studies:         {len(train_df):,}")
print(f"Studies with reports:            {len(train_df) - has_no_report:,}")
print(f"Studies with >= 1 NLP label:     {has_any_nlp:,}")
print(f"Studies with no NLP labels:      {len(train_df) - has_any_nlp:,}")
print(f"Gold-labeled studies:            {len(labeled_df):,}")

# ── NLP coverage per target ──────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"{'Target':20s} {'Mentioned':>10s} {'Positive':>10s} {'Negative':>10s} {'No mention':>10s}")
print("-" * 65)
for col in target_cols:
    mentioned = nlp_df[col].notna().sum()
    pos = (nlp_df[col] == 1).sum()
    neg = (nlp_df[col] == 0).sum()
    nan_count = nlp_df[col].isna().sum()
    print(f"{col:20s} {mentioned:>10,} {pos:>10,} {neg:>10,} {nan_count:>10,}")

# ── Prevalence comparison: Gold vs NLP ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

gold_prev = labeled_df[target_cols].mean().values
nlp_prev = nlp_df[target_cols].apply(lambda x: x.dropna().mean()).values

x = np.arange(len(target_cols))
w = 0.35
bars1 = ax.bar(x - w/2, gold_prev, w, label=f'Gold labels (n={len(labeled_df)})', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + w/2, nlp_prev, w, label=f'NLP-extracted (n={has_any_nlp})', color='#e74c3c', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(target_cols, rotation=45, ha='right')
ax.set_ylabel("Prevalence (fraction positive)")
ax.set_title("Label Prevalence — Gold Standard vs NLP Extraction", fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\nNote: Gold labels are biased toward pathological cases (avg 4.14 abnormalities/study).")
print("NLP prevalence is likely closer to the true population distribution.")

In [ ]:
# ── Validate NLP labels against gold standard ────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get NLP predictions for the 58 gold-labeled studies
gold_idx = labeled_df.index
nlp_for_gold = nlp_df.loc[gold_idx]

print("NLP Label Extraction — Validation Against 58 Gold Labels")
print("=" * 80)
print(f"{'Target':20s} {'Acc':>6s} {'Prec':>6s} {'Recall':>6s} {'F1':>6s}  {'TP':>4s} {'FP':>4s} {'FN':>4s} {'TN':>4s}  {'Cov':>5s}")
print("-" * 80)

overall_correct = 0
overall_total = 0

for col in target_cols:
    gold = labeled_df[col].values
    nlp = nlp_for_gold[col].values
    
    # Only evaluate where NLP made a prediction (not NaN)
    mask = ~np.isnan(nlp)
    coverage = mask.sum() / len(gold)
    
    if mask.sum() == 0:
        print(f"{col:20s} {'—':>6s} {'—':>6s} {'—':>6s} {'—':>6s}  {'—':>4s} {'—':>4s} {'—':>4s} {'—':>4s}  {coverage:>5.0%}")
        continue
    
    g, n = gold[mask], nlp[mask].astype(int)
    
    tp = ((g == 1) & (n == 1)).sum()
    fp = ((g == 0) & (n == 1)).sum()
    fn = ((g == 1) & (n == 0)).sum()
    tn = ((g == 0) & (n == 0)).sum()
    
    acc = accuracy_score(g, n)
    prec = precision_score(g, n, zero_division=0)
    rec = recall_score(g, n, zero_division=0)
    f1 = f1_score(g, n, zero_division=0)
    
    overall_correct += (tp + tn)
    overall_total += len(g)
    
    print(f"{col:20s} {acc:>6.1%} {prec:>6.1%} {rec:>6.1%} {f1:>6.1%}  {tp:>4d} {fp:>4d} {fn:>4d} {tn:>4d}  {coverage:>5.0%}")

overall_acc = overall_correct / overall_total if overall_total > 0 else 0
print("-" * 80)
print(f"{'OVERALL':20s} {overall_acc:>6.1%}")
print(f"\nCov = coverage (fraction of gold studies where NLP made a prediction)")
print(f"Low coverage targets may need more pattern variants or multilingual support.")

In [ ]:
# ── Merge: Gold labels override NLP, NLP fills the rest ──────────────────
# Start with NLP labels for all studies
labeled_df_expanded = train_df[['StudyInstanceUID']].copy()
for col in target_cols:
    labeled_df_expanded[col] = nlp_df[col]

# Override with gold labels where available (gold is ground truth)
for col in target_cols:
    gold_mask = labeled_df.index
    labeled_df_expanded.loc[gold_mask, col] = labeled_df[col].values

# Drop rows where ALL targets are NaN (no info at all)
labeled_df_expanded = labeled_df_expanded.dropna(subset=target_cols, how='all').copy()

# For training, fill remaining NaN with 0 (conservative: assume normal if not mentioned)
# Keep a record of which were imputed for potential label-smoothing later
nlp_imputed_mask = labeled_df_expanded[target_cols].isna()
n_imputed = nlp_imputed_mask.sum().sum()
labeled_df_expanded[target_cols] = labeled_df_expanded[target_cols].fillna(0)

print("=" * 60)
print("EXPANDED DATASET SUMMARY")
print("=" * 60)
print(f"  Gold-labeled studies:      {len(labeled_df):>6,}")
print(f"  NLP-labeled studies:       {len(labeled_df_expanded) - len(labeled_df):>6,}")
print(f"  Total expanded dataset:    {len(labeled_df_expanded):>6,}")
print(f"  Expansion factor:          {len(labeled_df_expanded)/len(labeled_df):>6.1f}x")
print(f"  NaN cells filled with 0:   {n_imputed:>6,} ({n_imputed/(len(labeled_df_expanded)*len(target_cols))*100:.1f}%)")

# Show new prevalence
print(f"\nExpanded dataset prevalence:")
for col in target_cols:
    prev = labeled_df_expanded[col].mean()
    gold_prev_val = labeled_df[col].mean()
    print(f"  {col:20s}: {prev:>6.1%}  (was {gold_prev_val:>6.1%} in gold)")

print(f"\n>>> labeled_df_expanded is ready for training ({len(labeled_df_expanded):,} studies, {len(target_cols)} targets)")

---
# 3. Series & Study Analysis

Each patient study contains multiple MRI **series** — different imaging sequences acquired in different anatomical planes. Understanding this structure is key to building the right model architecture.

In [ ]:
# Series per study
series_per_study = train_series_df.groupby('StudyInstanceUID').size()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of series per study
axes[0].hist(series_per_study.values, bins=range(1, series_per_study.max()+2),
             color='steelblue', edgecolor='white', linewidth=1, align='left')
axes[0].set_xlabel("Number of series per study", fontsize=12)
axes[0].set_ylabel("Number of studies", fontsize=12)
axes[0].set_title("How many MRI series per patient?", fontsize=13, fontweight='bold')
axes[0].axvline(series_per_study.median(), color='red', linestyle='--', label=f'Median: {series_per_study.median():.0f}')
axes[0].legend()

# Series descriptions — what types of MRI sequences are used?
desc_counts = train_series_df['Anatomical_Plane'].value_counts().head(20)
axes[1].barh(range(len(desc_counts)), desc_counts.values, color='teal')
axes[1].set_yticks(range(len(desc_counts)))
axes[1].set_yticklabels(desc_counts.index, fontsize=9)
axes[1].set_xlabel("Count", fontsize=12)
axes[1].set_title("Anatomical Plane Distribution", fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\nSeries per study — min: {series_per_study.min()}, max: {series_per_study.max()}, "
      f"mean: {series_per_study.mean():.1f}, median: {series_per_study.median():.0f}")
print(f"Unique series descriptions: {train_series_df['Anatomical_Plane'].nunique()}")

In [ ]:
# Anatomical plane analysis
# The Anatomical_Plane column from the CSV already classifies planes
# For DICOM metadata, we also define a helper to classify from SeriesDescription
def classify_plane(desc):
    """Map series/plane description to standard anatomical plane name."""
    if pd.isna(desc):
        return 'Unknown'
    desc = str(desc).lower().strip()
    if desc in ('sagittal', 'sag'):
        return 'Sagittal'
    elif desc in ('coronal', 'cor'):
        return 'Coronal'
    elif desc in ('axial', 'ax', 'transverse', 'tra'):
        return 'Axial'
    elif 'sag' in desc:
        return 'Sagittal'
    elif 'cor' in desc:
        return 'Coronal'
    elif 'ax' in desc or 'tra' in desc:
        return 'Axial'
    else:
        return desc.title()  # Use as-is for unknown planes

train_series_df['plane'] = train_series_df['Anatomical_Plane'].apply(classify_plane)

plane_counts = train_series_df['plane'].value_counts()
print(f"Plane values: {dict(plane_counts)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plane distribution
colors_plane = {'Sagittal': '#3498db', 'Coronal': '#e74c3c', 'Axial': '#2ecc71', 'Other': '#95a5a6', 'Unknown': '#bdc3c7'}
axes[0].pie(plane_counts.values, labels=plane_counts.index, autopct='%1.1f%%',
            colors=[colors_plane.get(p, '#95a5a6') for p in plane_counts.index],
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title("Distribution of Anatomical Planes", fontsize=13, fontweight='bold')

# Planes per study — does every study have all 3 planes?
planes_per_study = train_series_df.groupby('StudyInstanceUID')['plane'].apply(lambda x: set(x))
plane_combos = planes_per_study.apply(lambda x: ', '.join(sorted(x))).value_counts().head(10)
axes[1].barh(range(len(plane_combos)), plane_combos.values, color='mediumpurple')
axes[1].set_yticks(range(len(plane_combos)))
axes[1].set_yticklabels(plane_combos.index, fontsize=9)
axes[1].set_xlabel("Number of studies", fontsize=12)
axes[1].set_title("Most common plane combinations per study", fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Studies with all 3 main planes
has_all_3 = planes_per_study.apply(lambda x: {'Sagittal', 'Coronal', 'Axial'}.issubset(x)).sum()
print(f"\nStudies with all 3 planes (Sag+Cor+Ax): {has_all_3:,} / {len(planes_per_study):,} ({has_all_3/len(planes_per_study)*100:.1f}%)")

---
# 4. DICOM Metadata Mining

This is where it gets interesting. DICOM files contain rich metadata beyond just the pixel data. Let's mine this to understand the imaging parameters, which are crucial for preprocessing decisions.

We'll sample DICOM files to extract:
- **Pixel Spacing** — physical size of each pixel (mm)
- **Slice Thickness** — distance between slices (mm)  
- **Rows/Columns** — image dimensions
- **Manufacturer** — scanner brand
- **Magnetic Field Strength** — 1.5T vs 3T
- **Sequence Name** — pulse sequence type
- **Window Center/Width** — display windowing

In [ ]:
%%time
# Sample DICOM metadata from training set
# We sample 1 DICOM per series for speed (metadata is consistent within a series)

import random
random.seed(42)

dcm_metadata = []
sample_studies = random.sample(train_studies, min(300, len(train_studies)))

for study_dir in sample_studies:
    study_id = study_dir.name
    if not study_dir.is_dir():
        continue
    for series_dir in study_dir.iterdir():
        if not series_dir.is_dir():
            continue
        series_id = series_dir.name
        dcm_files = list(series_dir.glob("*.dcm"))
        if not dcm_files:
            continue
        
        # Read one DICOM file per series for metadata
        dcm = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
        
        row = {
            'StudyInstanceUID': study_id,
            'SeriesInstanceUID': series_id,
            'n_slices': len(dcm_files),
            'rows': getattr(dcm, 'Rows', None),
            'columns': getattr(dcm, 'Columns', None),
            'pixel_spacing_r': float(dcm.PixelSpacing[0]) if hasattr(dcm, 'PixelSpacing') else None,
            'pixel_spacing_c': float(dcm.PixelSpacing[1]) if hasattr(dcm, 'PixelSpacing') else None,
            'slice_thickness': float(dcm.SliceThickness) if hasattr(dcm, 'SliceThickness') else None,
            'manufacturer': getattr(dcm, 'Manufacturer', 'Unknown'),
            'model': getattr(dcm, 'ManufacturerModelName', 'Unknown'),
            'field_strength': float(dcm.MagneticFieldStrength) if hasattr(dcm, 'MagneticFieldStrength') else None,
            'sequence_name': getattr(dcm, 'SequenceName', 'Unknown'),
            'series_description': getattr(dcm, 'SeriesDescription', 'Unknown'),
            'laterality': getattr(dcm, 'Laterality', getattr(dcm, 'ImageLaterality', 'Unknown')),
            'bits_stored': getattr(dcm, 'BitsStored', None),
            'window_center': dcm.WindowCenter if hasattr(dcm, 'WindowCenter') else None,
            'window_width': dcm.WindowWidth if hasattr(dcm, 'WindowWidth') else None,
            'photometric': getattr(dcm, 'PhotometricInterpretation', 'Unknown'),
        }
        dcm_metadata.append(row)

dcm_df = pd.DataFrame(dcm_metadata)
print(f"Sampled DICOM metadata from {dcm_df['StudyInstanceUID'].nunique()} studies, {len(dcm_df)} series")
display(dcm_df.head())

In [ ]:
# Image dimensions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Rows x Columns scatter
ax = axes[0, 0]
ax.scatter(dcm_df['columns'], dcm_df['rows'], alpha=0.3, s=10, c='steelblue')
ax.set_xlabel("Columns (width)", fontsize=12)
ax.set_ylabel("Rows (height)", fontsize=12)
ax.set_title("Image Dimensions (Rows × Columns)", fontsize=13, fontweight='bold')

# Resolution distribution
dim_counts = dcm_df.apply(lambda r: f"{int(r['rows'])}×{int(r['columns'])}" if pd.notna(r['rows']) else 'Unknown', axis=1)
top_dims = dim_counts.value_counts().head(10)
ax = axes[0, 1]
ax.barh(range(len(top_dims)), top_dims.values, color='coral')
ax.set_yticks(range(len(top_dims)))
ax.set_yticklabels(top_dims.index)
ax.set_xlabel("Count", fontsize=12)
ax.set_title("Top 10 Image Resolutions", fontsize=13, fontweight='bold')
ax.invert_yaxis()

# Pixel spacing
ax = axes[1, 0]
valid_ps = dcm_df.dropna(subset=['pixel_spacing_r'])
ax.hist(valid_ps['pixel_spacing_r'], bins=50, color='seagreen', edgecolor='white', alpha=0.8)
ax.set_xlabel("Pixel Spacing (mm)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Pixel Spacing Distribution", fontsize=13, fontweight='bold')
ax.axvline(valid_ps['pixel_spacing_r'].median(), color='red', linestyle='--', 
           label=f"Median: {valid_ps['pixel_spacing_r'].median():.3f} mm")
ax.legend()

# Slice thickness
ax = axes[1, 1]
valid_st = dcm_df.dropna(subset=['slice_thickness'])
ax.hist(valid_st['slice_thickness'], bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
ax.set_xlabel("Slice Thickness (mm)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Slice Thickness Distribution", fontsize=13, fontweight='bold')
ax.axvline(valid_st['slice_thickness'].median(), color='red', linestyle='--',
           label=f"Median: {valid_st['slice_thickness'].median():.1f} mm")
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nPixel spacing range: {valid_ps['pixel_spacing_r'].min():.3f} – {valid_ps['pixel_spacing_r'].max():.3f} mm")
print(f"Slice thickness range: {valid_st['slice_thickness'].min():.1f} – {valid_st['slice_thickness'].max():.1f} mm")

In [ ]:
# Manufacturer and scanner analysis
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Manufacturers
mfr_counts = dcm_df['manufacturer'].value_counts().head(10)
axes[0].barh(range(len(mfr_counts)), mfr_counts.values, color='steelblue')
axes[0].set_yticks(range(len(mfr_counts)))
axes[0].set_yticklabels(mfr_counts.index, fontsize=9)
axes[0].set_xlabel("Count")
axes[0].set_title("Scanner Manufacturers", fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

# Field strength
fs = dcm_df['field_strength'].dropna()
if len(fs) > 0:
    fs_counts = fs.round(1).value_counts().sort_index()
    axes[1].bar(fs_counts.index.astype(str), fs_counts.values, color='coral')
    axes[1].set_xlabel("Field Strength (Tesla)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Magnetic Field Strength", fontsize=13, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, "No field strength data", ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title("Magnetic Field Strength", fontsize=13, fontweight='bold')

# Bits stored
bits_counts = dcm_df['bits_stored'].value_counts().sort_index()
axes[2].bar(bits_counts.index.astype(str), bits_counts.values, color='mediumpurple')
axes[2].set_xlabel("Bits Stored")
axes[2].set_ylabel("Count")
axes[2].set_title("Bits Per Pixel", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Laterality
print(f"\nLaterality distribution:")
print(dcm_df['laterality'].value_counts().to_string())

In [ ]:
# Pixel spacing vs slice thickness — important for resampling decisions
dcm_df['plane'] = dcm_df['series_description'].apply(classify_plane)

fig, ax = plt.subplots(figsize=(10, 7))
for plane, color in colors_plane.items():
    mask = dcm_df['plane'] == plane
    if mask.sum() > 0:
        sub = dcm_df[mask].dropna(subset=['pixel_spacing_r', 'slice_thickness'])
        ax.scatter(sub['pixel_spacing_r'], sub['slice_thickness'], 
                  alpha=0.4, s=20, c=color, label=f'{plane} ({mask.sum()})')

ax.set_xlabel("Pixel Spacing (mm)", fontsize=12)
ax.set_ylabel("Slice Thickness (mm)", fontsize=12)
ax.set_title("Pixel Spacing vs Slice Thickness by Anatomical Plane", fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, None)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

---
# 5. Image Visualization

Let's look at actual knee MRI images across different planes, abnormalities, and windowing settings.

In [ ]:
def load_dicom_image(dcm_path):
    """Load a DICOM file and return pixel array as float."""
    dcm = pydicom.dcmread(str(dcm_path))
    img = dcm.pixel_array.astype(np.float32)
    # Apply rescale if present
    slope = getattr(dcm, 'RescaleSlope', 1)
    intercept = getattr(dcm, 'RescaleIntercept', 0)
    img = img * slope + intercept
    return img, dcm

def apply_window(img, center, width):
    """Apply DICOM windowing."""
    lower = center - width / 2
    upper = center + width / 2
    img = np.clip(img, lower, upper)
    img = (img - lower) / (upper - lower)
    return img

def get_middle_slice(series_dir):
    """Get the middle slice from a series directory."""
    dcm_files = sorted(series_dir.glob("*.dcm"))
    if not dcm_files:
        return None, None
    mid_idx = len(dcm_files) // 2
    return load_dicom_image(dcm_files[mid_idx])

print("Helper functions defined.")

In [ ]:
# Show sample images from different anatomical planes
# Pick a study that has all 3 planes
merged = train_series_df.merge(train_df, on='StudyInstanceUID')
merged['plane'] = merged['Anatomical_Plane'].apply(classify_plane)

# Find a study with all 3 main planes and some abnormalities
study_plane_counts = merged.groupby('StudyInstanceUID')['plane'].apply(lambda x: set(x))
good_studies = study_plane_counts[study_plane_counts.apply(lambda x: {'Sagittal', 'Coronal', 'Axial'}.issubset(x))].index

# Pick one with multiple abnormalities for visual interest
abnormal_studies = labeled_df[labeled_df['n_abnormalities'] >= 3]
candidates = list(set(good_studies) & set(abnormal_studies['StudyInstanceUID']))
if candidates:
    demo_study_id = candidates[0]
else:
    demo_study_id = good_studies[0] if len(good_studies) > 0 else train_studies[0].name

demo_study_dir = DATA_DIR / "train_series" / str(demo_study_id)

# Get labels for this study
study_labels = labeled_df[labeled_df['StudyInstanceUID'] == demo_study_id][target_cols].iloc[0]
positive_labels = [col for col in target_cols if study_labels[col] == 1]

print(f"Demo Study: {demo_study_id}")
print(f"Abnormalities: {positive_labels if positive_labels else 'Normal'}")
print(f"Number of series: {len(list(demo_study_dir.iterdir()))}")

# Show middle slice from each series in this study
series_dirs = sorted(demo_study_dir.iterdir())
n_series = len(series_dirs)
fig, axes = plt.subplots(1, min(n_series, 8), figsize=(4 * min(n_series, 8), 4))
if n_series == 1:
    axes = [axes]

for idx, series_dir in enumerate(series_dirs[:8]):
    img, dcm = get_middle_slice(series_dir)
    if img is None:
        continue
    
    desc = getattr(dcm, 'SeriesDescription', 'Unknown')
    plane = classify_plane(desc)
    
    # Normalize for display
    p1, p99 = np.percentile(img, [1, 99])
    img_display = np.clip((img - p1) / max(p99 - p1, 1), 0, 1)
    
    axes[idx].imshow(img_display, cmap='gray')
    axes[idx].set_title(f"{plane}\n{desc[:30]}", fontsize=9)
    axes[idx].axis('off')

fig.suptitle(f"Study {demo_study_id} — All Series (middle slice)\nLabels: {', '.join(positive_labels) if positive_labels else 'Normal'}",
             fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### Windowing Effects
MRI windowing controls the brightness/contrast mapping. The same raw pixel data can reveal different structures depending on the window settings. This is critical for preprocessing — should we use DICOM window metadata or percentile-based normalization?

In [ ]:
# Windowing comparison — same image with different windows
# Pick a sagittal series for demonstration
sag_series = merged[merged['plane'] == 'Sagittal'].iloc[0]
sag_dir = DATA_DIR / "train_series" / str(sag_series['StudyInstanceUID'])

# Find the sagittal series directory
for sd in sag_dir.iterdir():
    if sd.is_dir():
        dcm_files = sorted(sd.glob("*.dcm"))
        if dcm_files:
            test_dcm = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
            desc = getattr(test_dcm, 'SeriesDescription', '')
            if 'sag' in desc.lower():
                img, dcm = load_dicom_image(dcm_files[len(dcm_files)//2])
                break

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Raw (no windowing)
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f"Raw\nRange: [{img.min():.0f}, {img.max():.0f}]", fontsize=10)
axes[0].axis('off')

# Percentile-based (1st-99th)
p1, p99 = np.percentile(img, [1, 99])
axes[1].imshow(np.clip((img - p1) / max(p99 - p1, 1), 0, 1), cmap='gray')
axes[1].set_title(f"Percentile (1%-99%)\n[{p1:.0f}, {p99:.0f}]", fontsize=10)
axes[1].axis('off')

# DICOM window (if available)
wc = dcm.WindowCenter if hasattr(dcm, 'WindowCenter') else img.mean()
ww = dcm.WindowWidth if hasattr(dcm, 'WindowWidth') else img.std() * 4
if isinstance(wc, pydicom.multival.MultiValue): wc = float(wc[0])
if isinstance(ww, pydicom.multival.MultiValue): ww = float(ww[0])
wc, ww = float(wc), float(ww)
axes[2].imshow(apply_window(img, wc, ww), cmap='gray')
axes[2].set_title(f"DICOM Window\nC={wc:.0f}, W={ww:.0f}", fontsize=10)
axes[2].axis('off')

# High contrast (narrow window)
axes[3].imshow(apply_window(img, wc, ww * 0.5), cmap='gray')
axes[3].set_title(f"Narrow Window\nC={wc:.0f}, W={ww*0.5:.0f}", fontsize=10)
axes[3].axis('off')

fig.suptitle("Effect of Different Windowing on the Same MRI Slice", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Normal vs. Abnormal — Can You Spot the Difference?

Let's compare images from normal knees vs. knees with specific abnormalities. This helps build intuition for what the model needs to learn.

In [ ]:
# Compare Normal vs Abnormal for key conditions
conditions_to_show = ['ACL', 'Medial Meniscus', 'Effusion', 'Fracture']

fig, axes = plt.subplots(len(conditions_to_show), 4, figsize=(16, 4 * len(conditions_to_show)))

for row_idx, condition in enumerate(conditions_to_show):
    # Get 2 normal and 2 abnormal studies
    normal_ids = labeled_df[labeled_df[condition] == 0]['StudyInstanceUID'].values
    abnormal_ids = labeled_df[labeled_df[condition] == 1]['StudyInstanceUID'].values
    
    samples = []
    for label, ids in [('Normal', normal_ids), ('Normal', normal_ids), 
                       (condition, abnormal_ids), (condition, abnormal_ids)]:
        # Try to find a study with a sagittal series
        found = False
        for sid in np.random.RandomState(row_idx + len(samples)).permutation(ids)[:20]:
            study_dir = DATA_DIR / "train_series" / str(sid)
            if not study_dir.exists():
                continue
            for series_dir in study_dir.iterdir():
                if not series_dir.is_dir():
                    continue
                dcm_files = sorted(series_dir.glob("*.dcm"))
                if not dcm_files:
                    continue
                test_dcm = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
                desc = getattr(test_dcm, 'SeriesDescription', '')
                if 'sag' in desc.lower():
                    img, _ = load_dicom_image(dcm_files[len(dcm_files)//2])
                    samples.append((img, label, desc[:25]))
                    found = True
                    break
            if found:
                break
        if not found:
            samples.append((None, label, 'N/A'))
    
    for col_idx, (img, label, desc) in enumerate(samples):
        ax = axes[row_idx, col_idx]
        if img is not None:
            p1, p99 = np.percentile(img, [1, 99])
            img_display = np.clip((img - p1) / max(p99 - p1, 1), 0, 1)
            ax.imshow(img_display, cmap='gray')
        else:
            ax.text(0.5, 0.5, 'No image', ha='center', va='center', transform=ax.transAxes)
        
        color = '#2ecc71' if label == 'Normal' else '#e74c3c'
        ax.set_title(f"{label}", fontsize=10, color=color, fontweight='bold')
        ax.axis('off')
    
    # Row label
    axes[row_idx, 0].set_ylabel(condition, fontsize=12, fontweight='bold', rotation=0, 
                                 labelpad=60, va='center')

fig.suptitle("Normal vs. Abnormal — Sagittal View Comparison", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Slice-by-Slice Montage
An MRI series is a 3D volume. Let's visualize all slices in a single series to understand the spatial coverage.

In [ ]:
# Montage — all slices from one series
# Pick a series with a reasonable number of slices (20-40)
montage_study = demo_study_dir
montage_series = None
for sd in sorted(montage_study.iterdir()):
    if sd.is_dir():
        n_files = len(list(sd.glob("*.dcm")))
        if 15 <= n_files <= 40:
            montage_series = sd
            break

if montage_series is None:
    # Fallback: just use first series
    montage_series = list(montage_study.iterdir())[0]

dcm_files = sorted(montage_series.glob("*.dcm"))
n_slices = len(dcm_files)
ncols = 8
nrows = (n_slices + ncols - 1) // ncols

# Get series description
test_dcm = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
desc = getattr(test_dcm, 'SeriesDescription', 'Unknown')

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.5, nrows * 2.5))
axes = axes.ravel() if nrows > 1 else (axes if isinstance(axes, np.ndarray) else [axes])

for i in range(nrows * ncols):
    if i < n_slices:
        img, _ = load_dicom_image(dcm_files[i])
        p1, p99 = np.percentile(img, [1, 99])
        img_display = np.clip((img - p1) / max(p99 - p1, 1), 0, 1)
        axes[i].imshow(img_display, cmap='gray')
        axes[i].set_title(f"Slice {i+1}", fontsize=8)
    axes[i].axis('off')

fig.suptitle(f"Complete Series Montage — {desc} ({n_slices} slices)\nStudy: {demo_study_id}",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# 6. Slice-Level Analysis

How many slices does each series have? This matters because models need to decide how many slices to sample from each series — too few and you miss pathology, too many and training slows down.

In [ ]:
# Count slices per series across all training data
slice_counts = []
for study_dir in train_studies:
    if not study_dir.is_dir():
        continue
    study_id = study_dir.name
    for series_dir in study_dir.iterdir():
        if not series_dir.is_dir():
            continue
        n_dcm = len(list(series_dir.glob("*.dcm")))
        series_id = series_dir.name
        
        # Get plane from train_series_df
        match = train_series_df[train_series_df['SeriesInstanceUID'] == series_id]
        if len(match) > 0:
            desc = match.iloc[0]['Anatomical_Plane']
            plane = classify_plane(desc)
        else:
            plane = 'Unknown'
        
        slice_counts.append({
            'StudyInstanceUID': study_id,
            'SeriesInstanceUID': series_id,
            'n_slices': n_dcm,
            'plane': plane
        })

slices_df = pd.DataFrame(slice_counts)
print(f"Total series analyzed: {len(slices_df):,}")
print(f"\nSlice count statistics:")
print(slices_df['n_slices'].describe().to_string())

In [ ]:
# Slice count distributions by plane
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall distribution
ax = axes[0]
ax.hist(slices_df['n_slices'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel("Number of slices per series", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Slices per Series — Overall Distribution", fontsize=13, fontweight='bold')
ax.axvline(slices_df['n_slices'].median(), color='red', linestyle='--', 
           label=f"Median: {slices_df['n_slices'].median():.0f}")
ax.legend()

# By plane (violin plot)
ax = axes[1]
planes_to_plot = ['Sagittal', 'Coronal', 'Axial']
plot_data = slices_df[slices_df['plane'].isin(planes_to_plot)]
if len(plot_data) > 0:
    parts = ax.violinplot([plot_data[plot_data['plane'] == p]['n_slices'].values 
                          for p in planes_to_plot if (plot_data['plane'] == p).sum() > 0],
                         showmeans=True, showmedians=True)
    valid_planes = [p for p in planes_to_plot if (plot_data['plane'] == p).sum() > 0]
    ax.set_xticks(range(1, len(valid_planes) + 1))
    ax.set_xticklabels(valid_planes)
    ax.set_ylabel("Number of slices", fontsize=12)
    ax.set_title("Slices per Series by Anatomical Plane", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Summary stats by plane
print("\nSlices per series by plane:")
print(slices_df.groupby('plane')['n_slices'].agg(['count', 'mean', 'median', 'min', 'max']).to_string())

In [ ]:
# Pixel intensity distributions across planes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

random.seed(42)
for idx, (plane, ax) in enumerate(zip(['Sagittal', 'Coronal', 'Axial'], axes)):
    plane_series = slices_df[slices_df['plane'] == plane]
    if len(plane_series) == 0:
        ax.text(0.5, 0.5, f'No {plane} series', ha='center', transform=ax.transAxes)
        continue
    
    # Sample a few series and plot their intensity histograms
    sample_ids = plane_series.sample(min(10, len(plane_series)), random_state=42)
    for _, row in sample_ids.iterrows():
        series_path = DATA_DIR / "train_series" / str(row['StudyInstanceUID']) / str(row['SeriesInstanceUID'])
        dcm_files = sorted(series_path.glob("*.dcm"))
        if not dcm_files:
            continue
        img, _ = load_dicom_image(dcm_files[len(dcm_files)//2])
        ax.hist(img.ravel(), bins=100, alpha=0.3, density=True)
    
    ax.set_xlabel("Pixel Intensity")
    ax.set_ylabel("Density")
    ax.set_title(f"{plane} — Intensity Distributions", fontsize=12, fontweight='bold')

plt.suptitle("Pixel Intensity Distributions Across Planes (10 samples each)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# 7. Train vs Test Distribution

A quick sanity check — do the train and test sets look similar in terms of series counts and metadata?

In [ ]:
# Compare train vs test series metadata
test_series_df['plane'] = test_series_df['Anatomical_Plane'].apply(classify_plane)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Series per study
train_sps = train_series_df.groupby('StudyInstanceUID').size()
test_sps = test_series_df.groupby('StudyInstanceUID').size()

ax = axes[0]
ax.hist(train_sps, bins=range(1, max(train_sps.max(), test_sps.max())+2), 
        alpha=0.6, color='steelblue', label=f'Train (n={len(train_sps)})', density=True, align='left')
ax.hist(test_sps, bins=range(1, max(train_sps.max(), test_sps.max())+2),
        alpha=0.6, color='coral', label=f'Test (n={len(test_sps)})', density=True, align='left')
ax.set_xlabel("Series per study")
ax.set_ylabel("Density")
ax.set_title("Series per Study", fontsize=13, fontweight='bold')
ax.legend()

# Plane distribution comparison
ax = axes[1]
train_planes = train_series_df['plane'].value_counts(normalize=True)
test_planes = test_series_df['plane'].value_counts(normalize=True)
all_planes_union = sorted(set(train_planes.index) | set(test_planes.index))
x = np.arange(len(all_planes_union))
w = 0.35
ax.bar(x - w/2, [train_planes.get(p, 0) for p in all_planes_union], w, label='Train', color='steelblue')
ax.bar(x + w/2, [test_planes.get(p, 0) for p in all_planes_union], w, label='Test', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(all_planes_union, rotation=30)
ax.set_ylabel("Fraction")
ax.set_title("Plane Distribution", fontsize=13, fontweight='bold')
ax.legend()

# Series description overlap
ax = axes[2]
train_descs = set(train_series_df['Anatomical_Plane'].dropna().unique())
test_descs = set(test_series_df['Anatomical_Plane'].dropna().unique())
overlap = train_descs & test_descs
train_only = train_descs - test_descs
test_only = test_descs - train_descs

from matplotlib_venn import venn2
try:
    venn2([train_descs, test_descs], set_labels=('Train', 'Test'), ax=ax)
    ax.set_title("Series Description Overlap", fontsize=13, fontweight='bold')
except:
    ax.bar(['Train only', 'Both', 'Test only'], [len(train_only), len(overlap), len(test_only)],
           color=['steelblue', 'mediumpurple', 'coral'])
    ax.set_title("Series Description Overlap", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nTrain: {len(train_descs)} unique descriptions")
print(f"Test:  {len(test_descs)} unique descriptions")
print(f"Overlap: {len(overlap)}")
print(f"Test-only (unseen in train): {len(test_only)}")

---
# 8. Full DICOM Header Reference

For reference, here's the complete DICOM header from one sample file. This is useful for identifying additional metadata fields you might want to leverage.

In [ ]:
# Print full DICOM header from one sample file
sample_study = train_studies[0]
sample_series = list(sample_study.iterdir())[0]
sample_dcm_file = list(sample_series.glob("*.dcm"))[0]

dcm = pydicom.dcmread(str(sample_dcm_file))
print(f"DICOM Header — {sample_dcm_file.name}")
print("=" * 70)
for elem in dcm:
    if elem.tag.group != 0x7FE0:  # Skip pixel data
        print(f"  {elem.tag} {elem.keyword:35s} {str(elem.value)[:80]}")

---
# 9. Key Takeaways & Modeling Implications

### Data Structure
- Each **study** (patient) has multiple MRI **series** acquired in different anatomical planes
- The 3 main planes are **Sagittal**, **Coronal**, and **Axial** — different abnormalities are best visible in different planes
- Most studies have all 3 planes, but the exact number of series varies

### Label Characteristics
- **12 binary targets** — this is a **multi-label** classification problem
- **Only 58 gold-labeled studies** out of 4,407 — heavily biased toward pathological cases (avg 4.14 abnormalities per knee)
- **NLP label extraction** from radiology reports expands usable training data by ~75x
- **Significant class imbalance** — rare conditions like Fracture and Contusion need special handling (weighted loss, oversampling)
- **Strong co-occurrence patterns** — e.g., ACL tears often come with contusions and effusion. This suggests **multi-task learning** could help

### Report Mining Insights
- Reports are the primary label source — regex NLP achieves ~82% agreement with gold labels
- Negation handling is critical ("no ACL tear" vs "ACL tear")
- Some reports are multilingual — a dedicated multilingual pipeline could further improve coverage
- NaN-filling strategy matters: treating unmentioned conditions as negative (0) is conservative but imperfect

### Image Properties
- **Variable image dimensions** — resizing/padding needed
- **Variable pixel spacing** — consider resampling to uniform physical resolution
- **Different scanners and field strengths** — domain adaptation/augmentation may help
- **Percentile-based normalization** (1st-99th percentile) is generally safer than relying on DICOM window metadata

### Modeling Strategy Hints
1. **Multi-plane fusion**: The winning baseline (Pilkwang Kim, 524 votes) uses a "slot" system that maps each abnormality to its most relevant anatomical plane
2. **Slice sampling**: Most series have 20-40 slices. The baseline samples 3 slices per plane (GROUP=3) and stacks as RGB
3. **DINOv2 backbone**: Self-supervised ViT features transfer well to medical imaging
4. **Anatomical priors**: ACL/Meniscus → Sagittal, OA → Coronal, Effusion → all planes
5. **Label smoothing**: NLP-derived labels are noisy (~82% accuracy) — smooth toward 0.5 for NLP-only studies

---
**If this notebook was helpful, please give it an upvote!** It helps the community find useful resources.

Feel free to fork this notebook and build on top of it. Happy kaggling!

---
# 10. Baseline Model & Submission

Now let's turn our EDA insights into a working model. Thanks to our **NLP label extraction** above, we're training on ~4,000+ studies instead of just 58.

- **2.5D approach**: Stack 3 adjacent slices as RGB channels (the technique that wins RSNA competitions)
- **ResNet18 backbone**: Small, fast, pretrained on ImageNet
- **Multi-label BCE loss**: With class weights derived from expanded dataset prevalence
- **Percentile normalization**: Based on our windowing analysis (safer than DICOM metadata)
- **Sagittal-only**: We saw that sagittal is the most information-dense plane for most conditions

This baseline trains on Kaggle's free CPU and produces a submittable CSV.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

# Use CPU — expanded dataset is larger but still manageable on CPU
DEVICE = torch.device('cpu')
IMG_SIZE = 224
NUM_SLICES = 3  # 2.5D: stack 3 adjacent slices as RGB
BATCH_SIZE = 16
EPOCHS = 8  # More epochs — we now have ~4,000+ studies instead of 58
LR = 1e-4

print(f"Device: {DEVICE}")
print(f"Image size: {IMG_SIZE}, Slices per input: {NUM_SLICES}")
print(f"Training with NLP-expanded labels: {len(labeled_df_expanded):,} studies (was {len(labeled_df)} gold-only)")

In [ ]:
class KneeDataset(Dataset):
    """2.5D dataset: pick best sagittal series, sample 3 adjacent middle slices."""
    
    def __init__(self, df, series_df, data_dir, target_cols, is_test=False):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.data_dir = Path(data_dir)
        self.target_cols = target_cols
        self.is_test = is_test
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
        ])
        
    def __len__(self):
        return len(self.df)
    
    def _find_sagittal_series(self, study_id):
        """Find the best sagittal series for a study."""
        study_series = self.series_df[self.series_df['StudyInstanceUID'] == study_id]
        sag = study_series[study_series['Anatomical_Plane'].str.lower().str.contains('sag', na=False)]
        if len(sag) > 0:
            return sag.iloc[0]['SeriesInstanceUID']
        # Fallback: use first series
        if len(study_series) > 0:
            return study_series.iloc[0]['SeriesInstanceUID']
        return None
    
    def _load_2_5d(self, study_id, series_id):
        """Load 3 adjacent middle slices as a 3-channel image."""
        folder = 'test_series' if self.is_test else 'train_series'
        series_dir = self.data_dir / folder / str(study_id) / str(series_id)
        
        dcm_files = sorted(series_dir.glob("*.dcm")) if series_dir.exists() else []
        if not dcm_files:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE)
        
        mid = len(dcm_files) // 2
        indices = [max(0, mid - 1), mid, min(len(dcm_files) - 1, mid + 1)]
        
        channels = []
        for idx in indices:
            dcm = pydicom.dcmread(str(dcm_files[idx]))
            img = dcm.pixel_array.astype(np.float32)
            slope = getattr(dcm, 'RescaleSlope', 1)
            intercept = getattr(dcm, 'RescaleIntercept', 0)
            img = img * slope + intercept
            # Percentile normalization (from our EDA analysis)
            p1, p99 = np.percentile(img, [1, 99])
            img = np.clip((img - p1) / max(p99 - p1, 1), 0, 1)
            img = (img * 255).astype(np.uint8)
            channels.append(self.transform(img))
        
        return torch.cat(channels, dim=0)  # [3, H, W]
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        series_df = self.series_df[self.series_df['StudyInstanceUID'] == study_id]
        series_id = self._find_sagittal_series(study_id)
        
        img = self._load_2_5d(study_id, series_id)
        
        if self.is_test:
            return img, study_id
        else:
            labels = torch.tensor(row[self.target_cols].values.astype(np.float32))
            return img, labels

print("KneeDataset defined.")

In [ ]:
# Model: ResNet18 with multi-label head
class KneeModel(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()
        self.backbone = models.resnet18(weights='IMAGENET1K_V1')
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

# Compute class weights from EXPANDED dataset (not just 58 gold labels)
pos_counts = labeled_df_expanded[target_cols].sum()
neg_counts = len(labeled_df_expanded) - pos_counts
pos_weight = (neg_counts / pos_counts.clip(lower=1)).values
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)

print(f"Model: ResNet18 → {len(target_cols)} outputs")
print(f"Class weights computed from {len(labeled_df_expanded):,} expanded studies")
print(f"\nClass weights (neg/pos ratio):")
for col, w in zip(target_cols, pos_weight):
    print(f"  {col:20s}: {w:.1f}")

In [ ]:
# Train/val split using EXPANDED labels (gold + NLP)
train_data, val_data = train_test_split(labeled_df_expanded, test_size=0.10, random_state=42)

train_dataset = KneeDataset(train_data, train_series_df, DATA_DIR, target_cols)
val_dataset = KneeDataset(val_data, train_series_df, DATA_DIR, target_cols)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = KneeModel(num_classes=len(target_cols)).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS
)

print(f"Training on {len(train_data):,} studies, validating on {len(val_data):,} studies")
print(f"  (was {len(labeled_df)} total with gold-only — now {len(labeled_df_expanded):,} with NLP expansion)")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

import sys
best_auc = 0
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            val_preds.append(torch.sigmoid(logits).cpu().numpy())
            val_labels.append(labels.cpu().numpy())
    
    val_preds = np.concatenate(val_preds)
    val_labels = np.concatenate(val_labels)
    
    # Per-target AUC
    aucs = []
    for i, col in enumerate(target_cols):
        if val_labels[:, i].sum() > 0 and val_labels[:, i].sum() < len(val_labels):
            aucs.append(roc_auc_score(val_labels[:, i], val_preds[:, i]))
    mean_auc = np.mean(aucs) if aucs else 0
    
    sys.stdout.flush()
    print(f"Epoch {epoch+1}/{EPOCHS} — "
          f"Train Loss: {train_loss/len(train_loader):.4f}, "
          f"Val Loss: {val_loss/len(val_loader):.4f}, "
          f"Val AUC: {mean_auc:.4f}")
    
    if mean_auc > best_auc:
        best_auc = mean_auc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')

print(f"\nBest validation AUC: {best_auc:.4f}")

In [ ]:
# Generate submission
model.load_state_dict(torch.load('/kaggle/working/best_model.pth', weights_only=True))
model.eval()

test_dataset = KneeDataset(test_df, test_series_df, DATA_DIR, target_cols, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

all_preds = {}
with torch.no_grad():
    for imgs, study_ids in test_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy()
        for sid, pred in zip(study_ids, probs):
            sid = sid if isinstance(sid, str) else str(sid.item())
            all_preds[sid] = pred

# Build submission in competition format
submission = sample_sub.copy()
for i, col in enumerate(target_cols):
    submission[col] = submission['StudyInstanceUID'].map(
        lambda sid: all_preds.get(str(sid), np.full(len(target_cols), 0.5))[i]
    )

submission.to_csv('submission.csv', index=False)
print(f"Submission saved: {submission.shape}")
display(submission.head())

# Cleanup
del model, train_dataset, val_dataset, test_dataset
gc.collect()
pass  # no GPU cleanup needed

---
## What's Next? (Ideas to Improve This Baseline)

This baseline is deliberately minimal. Here's a ranked list of improvements by expected impact:

1. **Use DINOv2 backbone** — the top-scoring approach (see Pilkwang Kim's 524-vote notebook)
2. **Add coronal + axial planes** — our EDA showed different abnormalities are visible in different planes
3. **Site-grouped cross-validation** — random folds leak scanner identity and inflate AUC by ~0.05
4. **Improve NLP label quality** — multilingual patterns, sentence-level parsing, confidence scores
5. **Label smoothing for NLP labels** — NLP labels are ~82% accurate; smooth toward 0.5 for noisy studies
6. **More slices** — we only use 3 middle slices; try 5-7 or attention over all slices
7. **Multi-plane attention head** — let the model learn which plane matters for each abnormality
8. **Augmentation** — rigid transforms (rotation, translation, intensity scaling) — but NO horizontal flips (laterality matters!)

---
**If this notebook helped you, please upvote!** It helps the community find useful resources.

Fork this notebook to get started with your own experiments. Happy kaggling!